In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime
import time
import random

In [5]:
def scrape(url):
    options = webdriver.ChromeOptions()
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=options)

    driver.get(url)

    # Scroll until seeMoreCars_button disappears
    WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CLASS_NAME, "MuiGrid-grid-sm-6")))
    time.sleep(5)

    container = driver.find_element(By.CLASS_NAME, "MuiGrid-spacing-xs-3")
    number_entries = 1

    while number_entries != len(driver.find_element(By.CLASS_NAME, "MuiGrid-spacing-xs-3").find_elements(By.CLASS_NAME, "MuiGrid-grid-sm-6")):
        container = driver.find_element(By.CLASS_NAME, "MuiGrid-spacing-xs-3")
        number_entries = len(container.find_elements(By.CLASS_NAME, "MuiGrid-grid-sm-6"))
        driver.execute_script("arguments[0].scrollIntoView();", container.find_elements(By.CLASS_NAME, "MuiGrid-grid-sm-6")[-2])
        time.sleep(random.uniform(0.5, 1))
            
    html = driver.page_source
    driver.quit()

    soup = BeautifulSoup(html, 'html.parser')
    html = soup.find('div', class_='MuiGrid-spacing-xs-3')
    html = html.find_all('div', class_='MuiGrid-grid-sm-6' )

    return html

In [6]:
cars = []
for i in range(8):
    url = f"https://www.flexicar.es/segunda-mano/#/distintivo/selloC/distintivo/selloE/distintivo/sello0/precio_from/{(i+8)}000/precio_to/{(i+9)}000/cv_from/80/km_to/100000/year_from/2013/year_to/2022/sortBy/price/order/ASC"
    cars += scrape(url)
    print(len(cars))

182
489
945
1520
2103
2716
3321
3977


In [7]:
# # Open Chrome webdriver
# options = webdriver.ChromeOptions()
# prefs = {"profile.managed_default_content_settings.images": 2}
# options.add_experimental_option("prefs", prefs)

# driver = webdriver.Chrome(options=options)
# url = "https://www.flexicar.es/segunda-mano/#/distintivo/selloC/distintivo/selloE/distintivo/sello0/precio_from/6000/precio_to/30000/cv_from/80/km_to/100000/year_from/2013/year_to/2022/sortBy/price/order/ASC"

# driver.get(url)

# # Scroll until seeMoreCars_button disappears
# WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CLASS_NAME, "MuiGrid-grid-sm-6")))
# time.sleep(8)

# container = driver.find_element(By.CLASS_NAME, "MuiGrid-spacing-xs-3")
# number_entries = 0

# while number_entries != len(driver.find_element(By.CLASS_NAME, "MuiGrid-spacing-xs-3").find_elements(By.CLASS_NAME, "MuiGrid-grid-sm-6")):
#     container = driver.find_element(By.CLASS_NAME, "MuiGrid-spacing-xs-3")
#     number_entries = len(container.find_elements(By.CLASS_NAME, "MuiGrid-grid-sm-6"))
#     driver.execute_script("arguments[0].scrollIntoView();", container.find_elements(By.CLASS_NAME, "MuiGrid-grid-sm-6")[-1])
        
# html = driver.page_source
# driver.quit()

# soup = BeautifulSoup(html, 'html.parser')
# html = soup.find('div', class_='MuiGrid-spacing-xs-3')
# html = html.find_all('div', class_='MuiGrid-grid-sm-6' )

In [8]:
cars_data = []
for car in cars:
    url = car.find('a', class_='MuiLink-root')['href']
    url = 'https://www.flexicar.es/' + url

    title = car.find('h2', class_='MuiTypography-root').text
    details = car.find_all('div', class_='MuiBox-root')[-2]
    details = details.find_all('li')
    details = [detail.text for detail in details]
    year = int(details[0])
    mileage = details[1]
    mileage = int(re.sub(r'\D', '', mileage))
    fuel = details[2]
    transmission = details[3]

    details = car.find_all('p', class_='MuiTypography-root')
    details = [detail.text for detail in details]
    location = details.pop()
    offer = None
    iva = None
    if details[0][0] != 'D':
        full_price = details.pop(0)
        full_price = int(re.sub(r'\D', '', full_price))
    else:
        full_price = None
    monthly_rate = details[0]
    monthly_rate = int(re.sub(r'\D', '', monthly_rate))
    description = details[1]
    discounted_price = details[2]
    discounted_price = int(re.sub(r'\D', '', discounted_price))
    try:
        if details[3] == "Oferta":
            offer = details[3]
        else:
            iva = 'IVA Deducible'
    except IndexError:
        None
    try:
        if details[4] == "Oferta":
            offer = details[4]
        else:
            iva = 'IVA Deducible'
    except IndexError:
        None
    
    cars_data.append([title, full_price, description, discounted_price, monthly_rate, year, mileage, fuel, transmission, iva, offer, location, url])

df = pd.DataFrame(cars_data, columns=["title", "full_price", "description", "discounted_price", "monthly_rate", "year", "mileage", "fuel", "transmission", "iva_deducible", "offer", 'city', 'url'])

df.to_csv(f'data/flexicar/{datetime.now().strftime("%y.%m.%d")}_flexicar_raw.csv', index=False)


In [9]:
df.shape

(3977, 13)